# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haseebaabid/FlyRank_AI_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.

Row meaning  
One row = one pseudonymized client × one pseudonymized content item × one report date.

Table used  
The table is fact_content_daily_performance, partitioned by month=YYYY-MM.

Time window  
My slice is March 2026 (month=2026-03).

Label definition  
The label is whether CTR exceeds 0.10 (binary outcome derived from clicks ÷ impressions).

Exclusion rule  
I exclude branded queries and rows where gsc_data_available is false.*

In [9]:
con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1
""")



┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

In [10]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 5
""")


┌─────────────┬─────────────────────────┬──────────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │
│    date     │         varchar         │         varchar          │
├─────────────┼─────────────────────────┼──────────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │
└─────────────┴─────────────────────────┴──────────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

gsc_impressions, gsc_clicks, gsc_avg_position : search performance signals.

ga4_pageviews, scroll_events : engagement metrics.

Label:

CTR above 0.10 (binary outcome from clicks ÷ impressions).

Context:

report_date, month : time slice.

client_hash_id, content_hash_id : pseudonymized identifiers.

Excluded:

gsc_data_available : excluded because it leaks whether data exists.

ga4_total_engagement_sec : excluded because it’s post‑click behavior.

ai_meta

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END) AS rows_with_clicks,
        AVG(CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions,0)) AS avg_ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┬───────────────────────┐
│ total_rows │ rows_with_clicks │        avg_ctr        │
│   int64    │      int128      │        double         │
├────────────┼──────────────────┼───────────────────────┤
│    9841378 │           417981 │ 0.0030807484543084115 │
└────────────┴──────────────────┴───────────────────────┘

In [12]:
# Verification: impressions, clicks, CTR distribution for March 2026
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END) AS rows_with_clicks,
        AVG(CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions,0)) AS avg_ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")


┌────────────┬──────────────────┬───────────────────────┐
│ total_rows │ rows_with_clicks │        avg_ctr        │
│   int64    │      int128      │        double         │
├────────────┼──────────────────┼───────────────────────┤
│    9841378 │           417981 │ 0.0030807484543084163 │
└────────────┴──────────────────┴───────────────────────┘

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain check : Each row represents one client × one content item × one report date. The sample shows multiple content IDs for the same client on the same day, confirming the unit of analysis.

Row count + span : The March 2026 slice contains 9,841,378 rows, spanning from 2026‑03‑01 to 2026‑03‑31. This verifies the time window and total size of the dataset slice.

Missing values : GA4 fields (pageviews, sessions) are missing in ~3,018,741 rows, while gsc_clicks has no missing values. This confirms that GA4 data is incomplete and must be handled carefully.

Availability window : Out of 9.8M rows, 3,611,061 rows have gsc_data_available = TRUE. This shows that not all rows are usable for GSC‑based features, and availability filtering is required.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 5
""")



┌─────────────┬─────────────────────────┬──────────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │
│    date     │         varchar         │         varchar          │
├─────────────┼─────────────────────────┼──────────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │
└─────────────┴─────────────────────────┴──────────────────────────┘

In [14]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS start_date,
           MAX(report_date) AS end_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ start_date │  end_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [15]:
con.sql(f"""
    SELECT
        SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS missing_pageviews,
        SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_sessions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬──────────────────┬────────────────┐
│ missing_pageviews │ missing_sessions │ missing_clicks │
│      int128       │      int128      │     int128     │
├───────────────────┼──────────────────┼────────────────┤
│           3018741 │          3018741 │              0 │
└───────────────────┴──────────────────┴────────────────┘

In [16]:
con.sql(f"""
    SELECT COUNT(*) AS available_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬──────────┐
│ available_rows │ gsc_rows │
│     int64      │  int64   │
├────────────────┼──────────┤
│        9841378 │  3611061 │
└────────────────┴──────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced history : The dataset has uneven coverage across time. Some months have full GA4 + GSC data, while earlier months may only have GSC. This means historical comparisons are biased.

GSC‑only early rows : In the early part of the dataset, GA4 metrics are missing entirely. Those rows cannot be used for engagement features, limiting longitudinal analysis.

Window overlaps : Because data is partitioned by month, overlapping decision windows (e.g., rolling averages across months) can introduce leakage or misalignment. This restricts how far you can extend features across boundaries.

Never know user intent

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS missing_ga4
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

# Compare GSC vs GA4 coverage
con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
        SUM(CASE WHEN ga4_pageviews IS NOT NULL THEN 1 ELSE 0 END) AS ga4_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")



┌──────────┬──────────┐
│ gsc_rows │ ga4_rows │
│  int128  │  int128  │
├──────────┼──────────┤
│  3611061 │  6822637 │
└──────────┴──────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.